In [0]:
# ==================================================
# GOLD — Acidentes por Rodovia
# Pipeline PRF Acidentes de Trânsito
# ==================================================

from pyspark.sql.functions import col, round, sum, count, desc

# Lê da Silver
df = spark.table("workspace.default.silver_acidentes")

# Agrega por BR e UF
df_gold_rodovias = (df
    .groupBy("br", "uf")
    .agg(
        count("id").alias("total_acidentes"),
        sum("mortos").alias("total_mortos"),
        sum("feridos_graves").alias("total_feridos_graves"),
        sum("feridos_leves").alias("total_feridos_leves"),
        sum("feridos").alias("total_feridos"),
        sum("veiculos").alias("total_veiculos_envolvidos")
    )
    .withColumn("mortos_por_acidente", 
        round(col("total_mortos") / col("total_acidentes"), 3))
    .orderBy(desc("total_acidentes"))
)

# Grava Gold
(df_gold_rodovias.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_rodovias")
)

print("Gold rodovias gravada!")
df_gold_rodovias.show(10)